# 🚀 CAR-IMU: High-Performance CUDA Evaluation
This notebook is optimized for running 192-d LOSO cross-validation on A100/T4 GPUs using `evaluate_cuda.py`.

In [4]:
# 1. Setup Environment
!git clone -b wisdm2-data-branch https://github.com/aviral23032002/bio-pm-guided-synthetic-imu-generation.git
%cd bio-pm-guided-synthetic-imu-generation
!pip install h5py torch scikit-learn pandas matplotlib

Cloning into 'bio-pm-guided-synthetic-imu-generation'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 167 (delta 43), reused 156 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 9.71 MiB | 18.32 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/bio-pm-guided-synthetic-imu-generation/bio-pm-guided-synthetic-imu-generation


In [5]:
# 2. Mount Google Drive & Set Path
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

os.environ['BIOPM_PATH'] = '/content/drive/MyDrive/BIOPM'
biopm_root = os.environ['BIOPM_PATH']
print(f"🌍 Global BIOPM path: {biopm_root}")

🌍 Global BIOPM path: /content/drive/MyDrive/BIOPM


In [ ]:
# 3. Handle Data Loading (Unzip to local for max speed)
zip_path = os.path.join(biopm_root, 'car_imu_results_package.zip')

if os.path.exists(zip_path):
    print(f"📦 Unzipping to local runtime...")
    !unzip -q -o "{zip_path}" -d .
    
    # 4. Recover missing token_store.hdf5 from Drive if needed
    local_token_store = "results_wisdm_v2_6class/token_store.hdf5"
    drive_token_store = os.path.join(biopm_root, local_token_store)
    
    if not os.path.exists(local_token_store):
        if os.path.exists(drive_token_store):
            print("🔄 token_store.hdf5 missing from zip. Copying from Drive...")
            !mkdir -p results_wisdm_v2_6class
            !cp "{drive_token_store}" "{local_token_store}"
            print("✅ token_store.hdf5 recovered.")
        else:
            print(f"❌ ERROR: token_store.hdf5 not found on Drive at: {drive_token_store}")
    else:
        print("✅ token_store.hdf5 verified locally.")
    
    print("✨ Setup Ready!")
else:
    print("❌ ERROR: Zip not found on Drive. Please upload car_imu_results_package.zip to BIOPM folder.")

📦 Unzipping to local runtime...


In [ ]:
# 4. Run High-Speed CUDA Evaluation (192-d MLP)
real_tokens = "results_wisdm_v2_6class/token_store.hdf5"
syn_tokens  = "synthetic_tokens_v2_6class/synthetic_tokens.hdf5"

if os.path.exists(real_tokens):
    !python evaluate_cuda.py \
        --real "{real_tokens}" \
        --syn "{syn_tokens}"
else:
    print("❌ MISSING TOKENS. Check the folder structure in your zip.")

In [ ]:
# 5. Save results to Drive
!mkdir -p "$BIOPM_PATH/results_cuda_final"
!cp -r results_local/* "$BIOPM_PATH/results_cuda_final/"